RAG with webpage data extraction

In [ ]:
# !pip install beautifulsoup4 lxml

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

cwd = Path.cwd()
env_candidates = [
    cwd / ".env",
    cwd / "notebooks" / ".env",
    cwd.parent / "notebooks" / ".env",
]
env_file = next((p for p in env_candidates if p.exists()), None)
if env_file is None:
    raise FileNotFoundError("No .env found. Expected notebooks/.env with OPENAI_API_KEY.")
load_dotenv(env_file, override=True)

key = (os.getenv("OPENAI_API_KEY") or "").strip()
if not key or "paste_your_key" in key:
    raise ValueError(f"Set OPENAI_API_KEY in {env_file}.")

print(f"Loaded env from: {env_file}")
print("OPENAI_API_KEY configured: True")

In [2]:
from langchain_openai import ChatOpenAI

MAX_OUTPUT_TOKENS = 1000

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    max_tokens=MAX_OUTPUT_TOKENS,
)


In [ ]:
# Load webpage(s). Put any URL you want to read here.
from langchain_community.document_loaders import WebBaseLoader

urls = [
    "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
]

loader = WebBaseLoader(urls)
documents = loader.load()

print(f"Loaded {len(documents)} webpage(s)")
for doc in documents:
    print(doc.metadata.get("source"), "chars:", len(doc.page_content))

In [ ]:
# Text Splitting

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, add_start_index=True)

all_split_docs = text_splitter.split_documents(documents)

len(all_split_docs)  # Total number of chunks after splitting the documents


In [ ]:
# Embedding the chunks with OpenAI
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_one = embeddings.embed_query(all_split_docs[0].page_content)
vector_two = embeddings.embed_query(all_split_docs[1].page_content)

print(len(vector_one))
print(len(vector_two))

In [ ]:
# Vector store (new folder: OpenAI embeddings are 1536-dim, old Ollama index was 768)
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=all_split_docs,
    embedding=embeddings,
    persist_directory="./chroma_langchain_db_openai",
    collection_name="webpage_rag_openai",
)

In [ ]:
# Retrieve relevant chunks from the webpage(s)
from langchain_chroma import Chroma

vector_store = Chroma(
    persist_directory="./chroma_langchain_db_openai",
    embedding_function=embeddings,
    collection_name="webpage_rag_openai",
)

question = "What is retrieval-augmented generation?"
retrieved_docs = vector_store.similarity_search(question, k=3)

retrieved_docs

In [ ]:
# Generate an answer from the retrieved context

context = "\n\n".join(doc.page_content for doc in retrieved_docs)

prompt = f"""Answer the question using only the context below.
If the answer is not present in the context, say: I don't know based on the webpage.

Context:
{context}

Question: {question}
Answer:"""

response = llm.invoke(prompt)
print(response.content)

In [ ]:
# Retriever over the loaded webpage(s)
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

retriever.invoke("What is retrieval-augmented generation?")

In [ ]:
# Full RetrievalQA implementation

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Full RetrievalQA implementation
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

question = "What is retrieval-augmented generation?"

prompt = ChatPromptTemplate.from_template("""Use only the context below to answer the question.
If the answer is not in the context, say: I don't know based on the documents.
Keep the answer concise and do not add unsupported details.

Context:
{context}

Question: {question}
Answer:""")


def format_documents(documents):
    return "\n\n".join(document.page_content for document in documents)


retrieval_qa_chain = (
    {
        "context": retriever | format_documents,
        "question": lambda value: value,
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = retrieval_qa_chain.invoke(question)
print(answer)

prompt = ChatPromptTemplate.from_template("""Use only the context below to answer the question.
If the answer is not in the context, say: I don't know based on the documents.
Keep the answer concise and do not add unsupported details.

Context:
{context}

Question: {question}
Answer:""")


def format_documents(documents):
    return "\n\n".join(document.page_content for document in documents)


retrieval_qa_chain = (
    {
        "context": retriever | format_documents,
        "question": lambda value: value,
    }
    | prompt
    | llm
    | StrOutputParser()
)

answer = retrieval_qa_chain.invoke(question)
print(answer)

In [ ]:
# RetrivalQA

